In [1]:
import pandas as pd

In [2]:
import torch

In [3]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.5.1+cu121
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
from datasets import load_dataset
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from joblib import Parallel, delayed

dataset = load_dataset("empathetic_dialogues", trust_remote_code=True)

emotion_model = AutoModelForSequenceClassification.from_pretrained(
    "joeddav/distilbert-base-uncased-go-emotions-student"
).to(device)

emotion_tokenizer = AutoTokenizer.from_pretrained(
    "joeddav/distilbert-base-uncased-go-emotions-student"
)

go_to_plutchik = {
    'admiration': 'trust', 'amusement': 'joy', 'anger': 'anger',
    'annoyance': 'anger', 'approval': 'joy', 'caring': 'trust',
    'confusion': 'anticipation', 'curiosity': 'anticipation',
    'desire': 'anticipation', 'disappointment': 'sadness',
    'disapproval': 'disgust', 'disgust': 'disgust', 'embarrassment': 'anticipation',
    'excitement': 'joy', 'fear': 'fear', 'gratitude': 'trust', 'grief': 'sadness',
    'joy': 'joy', 'love': 'trust', 'nervousness': 'fear', 'optimism': 'joy',
    'pride': 'joy', 'realization': 'surprise', 'relief': 'anticipation',
    'remorse': 'sadness', 'sadness': 'sadness', 'surprise': 'surprise'
}


c:\Users\chadp\OneDrive\Documents\NLP Project\Empathetic_Response_Generator\chatbot-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import pandas as pd
import torch
from joblib import Parallel, delayed

def classify_emotion(text):
    inputs = emotion_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = emotion_model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=1)
    top_class = torch.argmax(probs, dim=1).item()
    go_emotion = emotion_model.config.id2label[top_class]
    return go_to_plutchik.get(go_emotion, "neutral")

def classify_emotion_batch(batch):
    return [classify_emotion(text) for text in batch]

def parallelize_emotion_classification(df, batch_size=32):
    batches = [df['utterance'].iloc[i:i + batch_size] for i in range(0, len(df), batch_size)]
    results = Parallel(n_jobs=-1)(delayed(classify_emotion_batch)(batch) for batch in batches)
    emotions = [emotion for sublist in results for emotion in sublist]
    df['emotion'] = emotions
    return df

def save_files(df):
    df_grouped = df.groupby('conv_id')[['utterance']].agg(list)
    data = []

    for idx in range(len(df_grouped)):
        utterances = df_grouped.iloc[idx]['utterance']

        for i in range(1, len(utterances), 2):
            input_text = utterances[i - 1]
            label = utterances[i]

            input_emotion = classify_emotion(input_text)
            label_emotion = classify_emotion(label)

            data.append({
                'conversation': {
                    'input_text': input_text,
                    'label': label,
                    'input_emotion': input_emotion,
                    'label_emotion': label_emotion
                }
            })

    out_df = pd.DataFrame(columns=['conversation'], data=data)
    return out_df

In [9]:
raw_train_data = dataset['train']
train_data = pd.DataFrame(raw_train_data)
raw_test_data = dataset['test']
test_data = pd.DataFrame(raw_test_data)
raw_val_data = dataset['validation']
val_data = pd.DataFrame(raw_val_data)

train_data['utterance'] = train_data['utterance'].str.replace("_comma_", ",")
val_data['utterance'] = val_data['utterance'].str.replace("_comma_", ",")
test_data['utterance'] = test_data['utterance'].str.replace("_comma_", ",")

In [10]:
train_convs = save_files(train_data)
val_convs = save_files(val_data)
test_convs = save_files(test_data)

train_convs.to_pickle('./dataset_emotions/train_emotions.pkl')
val_convs.to_pickle('./dataset_emotions/val_emotions.pkl')
test_convs.to_pickle('./dataset_emotions/test_emotions.pkl')

In [11]:
import pandas as pd

train_data = pd.read_pickle('./dataset_emotions/train_emotions.pkl')

pd.set_option('display.max_colwidth', None)

print(train_data.head(5))

                                                                                                                                                                                                                                                                                                                                                conversation
0  {'input_text': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.', 'label': 'Was this a friend you were in love with, or just a best friend?', 'input_emotion': 'joy', 'label_emotion': 'anticipation'}
1                                                                                                                                                                                                         {'input_text': 'This was a best friend. I miss her.', 'label': 'Where has she gone?', 'input_emotion

In [13]:
print(train_convs.shape)
print(val_convs.shape)
print(test_convs.shape)

(36629, 1)
(5712, 1)
(5242, 1)
